# One-vs-Rest — A Yes/No Scorecard per Type

**This notebook is the runnable twin of [`binary-logistic-walkthrough.html`](binary-logistic-walkthrough.html).** It builds one yes/no model ("is this player a grinder?") with the classic scorecard tools — **Information Value, WoE bins, points, KS** — then loops to do all 9 types, which *is* One-vs-Rest.

**How to read it:** each step has a short plain-English note, then a clean code cell. Notes carry the story; code stays short.

**Related:** [Multinomial notebook](multinomial-notebook.ipynb) · [MN vs OvR explainer](mn-vs-ovr.html)

---

In [ ]:
# Setup (safe to re-run)
# !pip install pandas scikit-learn matplotlib
import json
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt

CANDIDATES = [Path('../../data/players.json'), Path('data/players.json'), Path('players.json')]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH, 'Could not find players.json — set DATA_PATH manually.'
print('Using:', DATA_PATH.resolve())

## Step 0 — Build the answer column

**What / why:** same as the multinomial notebook — turn the documented ID-range rules into a type label per player, so we have answers to learn from.

In [ ]:
# Step 0 — load + label
raw = json.loads(DATA_PATH.read_text(encoding='utf-8'))
players = raw['players'] if isinstance(raw, dict) and 'players' in raw else raw
df = pd.DataFrame(players)

def archetype_from_id(pid):
    n = int(str(pid).split('-')[1])
    if   100 <= n <= 107: return 'new'
    elif 108 <= n <= 141: return 'recreational'
    elif 142 <= n <= 163: return 'regular'
    elif 164 <= n <= 175: return 'grinder'
    elif 176 <= n <= 183: return 'aggressive_predatory'
    elif 184 <= n <= 191: return 'promo_hunter'
    elif 192 <= n <= 197: return 'shared_device_household'
    elif 198 <= n <= 202: return 'cluster_member'
    elif 203 <= n <= 220: return 'healthy_anchor'
    else:                 return 'bot_like'

df['archetype'] = df['player_id'].map(archetype_from_id)
FEATURES = [c for c in ['registered_days_ago','lifetime_hands','avg_session_minutes','sessions_last_30d',
            'vpip','pfr','aggression_factor','avg_pot_size_bb','promo_redemptions_30d'] if c in df.columns]
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
print(f'{len(df)} players, {df.archetype.nunique()} types')

## Step 1 — The goal: one yes/no question

**What:** pick ONE type and make a yes/no target. We'll use **grinder** — every player is either a grinder (1) or not (0).

**Why:** yes/no is the simplest model and the building block of One-vs-Rest. Do this once per type and you've classified everyone.

In [ ]:
# Step 1 — build the yes/no target for one type
TARGET_TYPE = 'grinder'
df['is_target'] = (df['archetype'] == TARGET_TYPE).astype(int)
print(f'"{TARGET_TYPE}": {df.is_target.sum()} YES  /  {len(df)-df.is_target.sum()} NO')

## Step 3 — Rank the clues with Information Value (IV)

**What:** because the answer is yes/no, we can score how well each clue *separates* grinders from non-grinders. That score is **Information Value**.

**Why:** it tells you up front which clues carry signal and which are dead weight. *(Steps 2 & 4 — meet the data, fix blanks — are the same as the MN notebook, already done above.)*

> In production you'd use `OptBinning` / `toad` / `scorecardpy`; here we compute WoE and IV by hand so you can see exactly what they are.

In [ ]:
# Step 3 + 5 helper — bin a clue and compute WoE per bin + total IV (the scorecard core)
def woe_table(x, y, bins=4):
    d = pd.DataFrame({'x': x, 'y': y})
    d['bin'] = pd.qcut(d['x'], q=bins, duplicates='drop')
    g = d.groupby('bin', observed=True)['y'].agg(n='count', yes='sum')
    g['no'] = g['n'] - g['yes']
    g['dist_yes'] = (g['yes'] / g['yes'].sum()).replace(0, 1e-4)
    g['dist_no']  = (g['no']  / g['no'].sum()).replace(0, 1e-4)
    g['woe'] = np.log(g['dist_yes'] / g['dist_no'])
    g['iv_part'] = (g['dist_yes'] - g['dist_no']) * g['woe']
    return g, g['iv_part'].sum()

# Rank every clue by IV
iv_scores = {f: woe_table(df[f], df['is_target'])[1] for f in FEATURES}
iv_rank = pd.Series(iv_scores).sort_values(ascending=False)
print('Information Value per clue (higher = better separator):')
print(iv_rank.round(3))

## Step 5 — Bin a clue with WoE

**What:** chop the strongest clue into ranges (bins) and replace each range with a number (Weight of Evidence) saying how grinder-ish it is.

**Why:** binning tames outliers and bends, and turns a clue into a human-readable table of ranges.

In [ ]:
# Step 5 — show the WoE bins for the top clue
top_clue = iv_rank.index[0]
tbl, iv = woe_table(df[top_clue], df['is_target'])
print(f'WoE bins for "{top_clue}"  (IV = {iv:.2f})')
display(tbl[['n','yes','no','woe']].round(2))
tbl['woe'].plot(kind='bar', color='#33e0c4', figsize=(7,3),
                title=f'Higher WoE bin = more grinder-ish ({top_clue})'); plt.tight_layout(); plt.show()

## Steps 6–8 — Drop twins, split, train

**What:** keep non-overlapping clues, split study/quiz, fit one binary logistic regression.

**Why:** same golden rules as always. (For brevity we scale the raw clues here; the WoE-transformed version is the pure-scorecard path.)

In [ ]:
# Steps 6-8 — scale, split, train one yes/no model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = StandardScaler().fit_transform(df[FEATURES])
y = df['is_target'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=2000, class_weight='balanced')  # balanced helps the rarer 'yes'
clf.fit(X_tr, y_tr)
print('Trained one yes/no model for:', TARGET_TYPE)

## Step 9 — Read what it learned

**What:** one weight per clue — positive pushes toward "yes (grinder)."

**Why:** the cleanest possible readout — one number per clue, no baseline to track.

In [ ]:
# Step 9 — one weight per clue
w = pd.Series(clf.coef_[0], index=FEATURES).sort_values(ascending=False)
print(f'What makes a "{TARGET_TYPE}"? (+ pushes yes, - pushes no)')
print(w.round(2))

## Step 10 — Turn it into points

**What:** rescale the weights into round points so the model reads like a scorecard.

**Why:** a points table is something anyone can audit and trust — the FICO-style deliverable.

In [ ]:
# Step 10 — simple illustrative points scaling (points double the odds every 20 pts)
factor = 20 / np.log(2)
points = (w * factor).round().astype(int)
print('Illustrative points per clue (per 1 standard-deviation of the clue):')
print(points)

## Step 11 — How good is it? (separation & KS)

**What:** check that grinders score high and non-grinders score low on the hidden quiz set. **KS** is the biggest gap between the two groups.

**Why:** for a yes/no model, separation matters more than raw accuracy — especially when 'yes' is rare.

In [ ]:
# Step 11 — KS statistic on the quiz set
def ks_stat(y_true, y_prob):
    d = pd.DataFrame({'y': y_true, 'p': y_prob}).sort_values('p')
    cum_yes = (d.y == 1).cumsum() / max((d.y == 1).sum(), 1)
    cum_no  = (d.y == 0).cumsum() / max((d.y == 0).sum(), 1)
    return float((cum_yes - cum_no).abs().max())

prob_te = clf.predict_proba(X_te)[:, 1]
print(f'KS on quiz set: {ks_stat(y_te, prob_te):.2f}   (higher = cleaner separation)')
from sklearn.metrics import roc_auc_score
print(f'AUC on quiz set: {roc_auc_score(y_te, prob_te):.2f}')

## Step 12 — Pick the cutoff

**What:** choose the probability above which we say "yes." Move it to trade off catching more grinders vs mislabeling non-grinders.

**Why:** the right cutoff depends on what a mistake costs — and in a yes/no model you control it directly.

In [ ]:
# Step 12 — see the trade-off at a couple of cutoffs
from sklearn.metrics import confusion_matrix
for cut in (0.3, 0.5, 0.7):
    yhat = (prob_te >= cut).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, yhat, labels=[0,1]).ravel()
    print(f'cutoff {cut}:  caught {tp} grinders, missed {fn}, false alarms {fp}')

## Combine — do it 9 times = One-vs-Rest

**What:** repeat the yes/no model for every type, then for each player pick the type whose model is most confident.

**Why:** that collection of 9 yes/no scorecards *is* the One-vs-Rest classifier. Here scikit-learn runs the loop for us so we can compare it to the multinomial model.

In [ ]:
# Combine — OneVsRest over all 9 types, then score on a held-out set
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score

yA = df['archetype'].values
XA_tr, XA_te, yA_tr, yA_te = train_test_split(X, yA, test_size=0.20, random_state=42, stratify=yA)
ovr = OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight='balanced'))
ovr.fit(XA_tr, yA_tr)
print(f'One-vs-Rest accuracy (9 yes/no models): {accuracy_score(yA_te, ovr.predict(XA_te)):.0%}')
print(f'(Compare to the multinomial notebook\'s single-model accuracy.)')

---
## Done

You built a yes/no scorecard with IV, WoE bins, points, and KS — then looped it into a full One-vs-Rest classifier. This is the path that keeps every scorecard lever.

**Production tools** that automate the binning/IV/stepwise loop: `OptBinning`, `toad`, `scorecardpy`. See the **[MN vs OvR explainer](mn-vs-ovr.html)** for when to pick each approach.